# 3.37 — Nested Cross-Validation

Nested cross-validation estimates how well an entire model-selection procedure will behave on future data. The inner loop chooses hyperparameters using validation folds, while the outer loop keeps untouched test folds for an honest estimate of the tuned procedure's generalization risk.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build nested cross-validation one idea at a time. Run each cell in order and read the printed intermediate values — every piece of the selection and evaluation logic is visible. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, indexing, losses, and deterministic synthetic data.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for shuffled folds and noisy toy data.

### 1. Empirical risk: a score is an average loss

Cross-validation is built from the same primitive as empirical risk minimization: compute a loss on examples, then average it. The lesson's verified toy losses are `0.191`, `0.135`, and `0.454`. Their average is the raw fit term $R_S$, before any complexity cost or validation protocol is considered.

In [ ]:
losses_w = np.array([0.191, 0.135, 0.454])  # three per-example losses from the lesson text.
print("losses:", losses_w)  # inspect the raw pieces before averaging.
print("sum of losses:", round(float(losses_w.sum()), 3))  # 0.191 + 0.135 + 0.454.
assert round(float(losses_w.sum()), 3) == 0.780  # concrete arithmetic check from the lesson.

▶ What you'll see: three small losses whose sum is exactly `0.780`.

In [ ]:
risk_w = float(losses_w.mean())  # empirical risk is average loss, not total loss.
print("empirical risk R_S:", round(risk_w, 3))  # 0.780 / 3.
assert round(risk_w, 3) == 0.260  # verified lesson value.

plt.figure(figsize=(4.4, 3))
plt.bar(["ex 1", "ex 2", "ex 3"], losses_w, color="steelblue")
plt.axhline(risk_w, color="crimson", linestyle="--", label=f"mean={risk_w:.3f}")
plt.title("1: empirical risk is the average loss")
plt.ylabel("loss")
plt.legend()
plt.show()

▶ What you'll see: the dashed mean line sits at `0.260`, summarizing the three example losses.

*Why it's done this way:* averaging makes the score comparable across folds with different numbers of examples. A sum would make larger folds look worse just because they contain more terms; the mean estimates expected loss for one future example.

### 2. Selection should include the cost term

The lesson warns that the raw fit is not always the decision score. If a method has a complexity, regularization, or operational cost of `0.060`, the selection score is $R_S + cost = 0.320$. This keeps the procedure from preferring a flexible setting merely because it fits the current sample nicely.

In [ ]:
cost_w = 0.060  # complexity / regularization / operational cost from the lesson.
score_w = risk_w + cost_w  # final selection score for this setting.
print("raw risk:", round(risk_w, 3))
print("cost term:", round(cost_w, 3))
print("decision score:", round(score_w, 3))
assert round(score_w, 3) == 0.320  # 0.260 + 0.060.

▶ What you'll see: the raw `0.260` becomes a decision score of `0.320` after cost is included.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["raw risk", "+ cost", "score"], [risk_w, cost_w, score_w], color=["steelblue", "orange", "seagreen"])
plt.title("2: selection uses the full score")
plt.ylabel("score component")
plt.show()

▶ What you'll see: the final bar is the sum of the raw loss bar and the cost bar.

*Why it's done this way:* a hyperparameter search is an optimization problem, so the formula being optimized must match the real preference. If flexibility has a cost, adding it to the risk is how we express the bias-variance bargain numerically.

### 3. Validation gaps measure how convincing a win is

A tempting alternative reaches a decision score of `0.364`. The lower score still wins, but the gap is only `0.044`, about `12.1%` of the alternative score. Nested CV does not just ask "which score is smallest?"; it also forces us to ask whether the difference is large enough to survive resampling noise.

In [ ]:
alternative_w = 0.364  # more flexible alternative score from the lesson.
gap_w = alternative_w - score_w  # positive means the current setting is lower/better.
relative_gap_w = gap_w / alternative_w  # scale-free size of the win.
print("baseline score:", round(score_w, 3))
print("alternative score:", round(alternative_w, 3))
print("gap:", round(gap_w, 3))
print("relative gap:", round(relative_gap_w, 3))
assert round(gap_w, 3) == 0.044
assert round(relative_gap_w, 3) == 0.121

▶ What you'll see: the baseline wins by `0.044`, which is a `0.121` relative gap.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["baseline", "alternative"], [score_w, alternative_w], color=["seagreen", "gray"])
plt.plot([0, 1], [score_w, alternative_w], color="crimson", marker="o", label=f"gap={gap_w:.3f}")
plt.title("3: the validation gap is evidence")
plt.ylabel("lower is better")
plt.legend()
plt.show()

▶ What you'll see: the vertical difference is visible but modest, matching the small numerical gap.

*Why it's done this way:* validation estimates are noisy because folds are finite samples. A tiny improvement can be a lucky split; the gap is the evidence margin the new setting must keep when the data are resampled.

### 4. Inner CV selects; outer CV estimates that selection rule

Nested CV separates two jobs. The **inner loop** picks the hyperparameter on training-only folds. The **outer loop** then evaluates the chosen hyperparameter on a held-out fold that did not participate in selection. The outer score therefore estimates the full procedure: "given training data, tune inside it, then deploy the chosen model."

In [ ]:
inner_scores_w = np.array([[0.340, 0.300, 0.315],
                           [0.330, 0.318, 0.309],
                           [0.355, 0.325, 0.333]])  # rows=outer folds, cols=three candidate settings.
lambdas_w = np.array([0.0, 0.1, 1.0])  # candidate stabilizing strengths.
chosen_cols_w = np.argmin(inner_scores_w, axis=1)  # inner loop chooses lowest validation score per outer fold.
chosen_lambdas_w = lambdas_w[chosen_cols_w]  # translate chosen columns into hyperparameter values.
print("inner validation scores:\n", inner_scores_w)
print("chosen λ per outer fold:", chosen_lambdas_w)
assert np.allclose(chosen_lambdas_w, [0.1, 1.0, 0.1])

▶ What you'll see: each outer fold can choose a different λ because its inner training sample is different.

In [ ]:
outer_scores_w = np.array([0.322, 0.351, 0.305])  # held-out scores after inner tuning.
nested_estimate_w = float(outer_scores_w.mean())  # final nested CV estimate.
print("outer scores:", outer_scores_w)
print("nested CV estimate:", round(nested_estimate_w, 3))
assert round(nested_estimate_w, 3) == 0.326

plt.figure(figsize=(5, 3))
plt.plot(np.arange(1, 4), outer_scores_w, marker="o", color="purple")
plt.axhline(nested_estimate_w, color="black", linestyle="--", label=f"mean={nested_estimate_w:.3f}")
plt.title("4: outer folds estimate the tuned procedure")
plt.xlabel("outer fold")
plt.ylabel("held-out loss")
plt.xticks([1, 2, 3])
plt.legend()
plt.show()

▶ What you'll see: the nested estimate is the mean of the three outer held-out losses.

*Why it's done this way:* if the outer fold helped choose λ, it would no longer be future-like. Keeping selection inside the training portion makes the outer fold an honest test of the entire tuning workflow.

### 5. Non-nested CV is optimistically biased

A common mistake is to tune hyperparameters and report the same validation scores used for tuning. Because the minimum of noisy estimates tends to be too low, this non-nested estimate is optimistic. Nested CV fixes the bias by evaluating the chosen setting on outer data that the chooser never saw.

In [ ]:
flat_cv_scores_w = np.array([0.333, 0.301, 0.312])  # one CV table reused for both selection and reporting.
selected_flat_w = int(np.argmin(flat_cv_scores_w))  # non-nested picks the minimum score.
reported_non_nested_w = float(flat_cv_scores_w[selected_flat_w])  # then reports that same minimum.
print("flat CV scores:", flat_cv_scores_w)
print("selected λ:", lambdas_w[selected_flat_w])
print("reported non-nested score:", round(reported_non_nested_w, 3))
assert round(reported_non_nested_w, 3) == 0.301

▶ What you'll see: the reported number is simply the most flattering validation score in the table.

In [ ]:
optimism_w = nested_estimate_w - reported_non_nested_w  # how much lower the reused validation score looks.
print("nested estimate:", round(nested_estimate_w, 3))
print("optimism gap:", round(optimism_w, 3))
assert round(optimism_w, 3) == 0.025

plt.figure(figsize=(4.6, 3))
plt.bar(["non-nested report", "nested estimate"], [reported_non_nested_w, nested_estimate_w], color=["crimson", "seagreen"])
plt.title("5: reusing validation is optimistic")
plt.ylabel("estimated loss")
plt.show()

▶ What you'll see: the non-nested bar is lower by `0.025`, but that lower value is the biased one.

*Why it's done this way:* selection searches over random validation noise. Reporting the same minimum rewards luck; an untouched outer fold measures whether the selected rule works after the luck is gone.

### 6. Stabilization can win after the full comparison

The lesson's stabilizing knob reduces the score by 20%, giving `0.256`. The final decision compares the baseline `0.320`, the flexible alternative `0.364`, and the stabilized score `0.256`. The smallest is the setting to carry forward for this toy case.

In [ ]:
stable_w = 0.80 * score_w  # 20% reduction from the stabilizing knob.
final_scores_w = np.array([score_w, alternative_w, stable_w])  # baseline, flexible, stabilized.
labels_w = ["baseline", "flexible", "stabilized"]
winner_w = int(np.argmin(final_scores_w))  # lower score wins.
print("final scores:", dict(zip(labels_w, np.round(final_scores_w, 3))))
print("winner:", labels_w[winner_w])
assert round(stable_w, 3) == 0.256
assert labels_w[winner_w] == "stabilized"

▶ What you'll see: the stabilized option has the lowest final score, `0.256`.

In [ ]:
plt.figure(figsize=(5, 3))
colors_w = ["gray", "orange", "seagreen"]
plt.bar(labels_w, final_scores_w, color=colors_w)
plt.title("6: choose by the full decision score")
plt.ylabel("lower is better")
plt.show()

▶ What you'll see: the stabilized bar is shortest, so it wins this verified toy comparison.

*Why it's done this way:* nested CV estimates future performance of a complete selection procedure. The winner is not the prettiest training fragment; it is the option with the best full score after costs, gaps, and stability are made explicit.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses a
> handful of small numbers, prints every intermediate value with an inline `# ->` showing the
> result, draws one picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Fold losses average into one validation score

A fold score is not the sum of its losses; it is their average, so folds can be compared on a per-example scale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)  # seed the toy, even though the hand-picked data below are fixed.
t1_losses = np.array([0.20, 0.40, 0.10, 0.30, 0.50, 0.00])  # tiny validation losses  # -> [0.2, 0.4, 0.1, 0.3, 0.5, 0.0]
print("validation losses:", t1_losses.tolist())  # -> [0.2, 0.4, 0.1, 0.3, 0.5, 0.0]
t1_total = float(t1_losses.sum())  # add losses before averaging  # -> 1.5
print("sum of losses:", round(t1_total, 3))  # -> 1.5
t1_count = int(t1_losses.size)  # count examples in the fold  # -> 6
print("number of examples:", t1_count)  # -> 6
t1_score = float(t1_losses.mean())  # empirical fold score  # -> 0.25
print("mean validation score:", round(t1_score, 3))  # -> 0.25
assert round(t1_score, 3) == 0.250

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t1_count), t1_losses, color="steelblue")
plt.axhline(t1_score, color="crimson", linestyle="--", label="mean = 0.250")
plt.xlabel("validation example")
plt.ylabel("loss")
plt.title("Toy 1 · average the fold losses")
plt.legend()
plt.show()

▶ What you'll see: six small losses whose dashed mean line sits at `0.250`.

### ✍️ Toy 2 · Cost term changes the score being optimized

Nested model selection compares the full decision score, so a raw validation risk must pay any stated cost.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)  # seed the toy for reproducibility.
t2_losses = np.array([0.18, 0.22, 0.20, 0.24, 0.16, 0.20])  # six raw validation losses  # -> [0.18, 0.22, 0.2, 0.24, 0.16, 0.2]
print("raw losses:", t2_losses.tolist())  # -> [0.18, 0.22, 0.2, 0.24, 0.16, 0.2]
t2_risk = float(t2_losses.mean())  # average raw fit term  # -> 0.2
print("raw risk:", round(t2_risk, 3))  # -> 0.2
t2_cost = 0.05  # extra complexity or operational cost  # -> 0.05
print("cost term:", round(t2_cost, 3))  # -> 0.05
t2_score = t2_risk + t2_cost  # score used for selection  # -> 0.25
print("decision score:", round(t2_score, 3))  # -> 0.25
assert round(t2_score, 3) == 0.250

plt.figure(figsize=(4.6, 2.8))
plt.bar(["risk", "cost", "score"], [t2_risk, t2_cost, t2_score], color=["steelblue", "orange", "seagreen"])
plt.ylabel("score component")
plt.title("Toy 2 · fit plus cost")
plt.show()

▶ What you'll see: the final score bar is `0.250`, not the raw risk bar at `0.200`.

### ✍️ Toy 3 · Validation gaps need a scale

A lower mean validation score wins, but the absolute gap and relative gap say how convincing the win is.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)  # seed the toy for reproducibility.
t3_baseline = np.array([0.30, 0.32, 0.31, 0.33, 0.29, 0.31])  # baseline fold scores  # -> [0.3, 0.32, 0.31, 0.33, 0.29, 0.31]
print("baseline fold scores:", t3_baseline.tolist())  # -> [0.3, 0.32, 0.31, 0.33, 0.29, 0.31]
t3_alternative = np.array([0.34, 0.35, 0.33, 0.36, 0.34, 0.35])  # alternative fold scores  # -> [0.34, 0.35, 0.33, 0.36, 0.34, 0.35]
print("alternative fold scores:", t3_alternative.tolist())  # -> [0.34, 0.35, 0.33, 0.36, 0.34, 0.35]
t3_base_mean = float(t3_baseline.mean())  # baseline average score  # -> 0.31
print("baseline mean:", round(t3_base_mean, 3))  # -> 0.31
t3_alt_mean = float(t3_alternative.mean())  # alternative average score  # -> 0.345
print("alternative mean:", round(t3_alt_mean, 3))  # -> 0.345
t3_gap = t3_alt_mean - t3_base_mean  # absolute lower-is-better gap  # -> 0.035
t3_relative_gap = t3_gap / t3_alt_mean  # scale-free evidence gap  # -> 0.101449...
print("absolute gap:", round(t3_gap, 3))  # -> 0.035
print("relative gap:", round(t3_relative_gap, 3))  # -> 0.101
assert round(t3_gap, 3) == 0.035
assert round(t3_relative_gap, 3) == 0.101

plt.figure(figsize=(4.8, 2.8))
plt.bar(["baseline", "alternative"], [t3_base_mean, t3_alt_mean], color=["seagreen", "gray"])
plt.plot([0, 1], [t3_base_mean, t3_alt_mean], color="crimson", marker="o", label="gap = 0.035")
plt.ylabel("mean validation score")
plt.title("Toy 3 · measure the win")
plt.legend()
plt.show()

▶ What you'll see: the baseline is lower, but only by `0.035` or about `10.1%` of the alternative.

### ✍️ Toy 4 · Inner CV selects a hyperparameter inside each outer fold

Each row is one outer training set; the inner loop picks the smallest validation score in that row.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)  # seed the toy for reproducibility.
t4_lambdas = np.array([0.01, 0.10, 1.00])  # candidate regularization strengths  # -> [0.01, 0.1, 1.0]
print("lambda grid:", t4_lambdas.tolist())  # -> [0.01, 0.1, 1.0]
t4_inner_scores = np.array([[0.44, 0.36, 0.39], [0.31, 0.34, 0.33], [0.41, 0.38, 0.37]])  # rows=outer folds  # -> [[0.44, 0.36, 0.39], [0.31, 0.34, 0.33], [0.41, 0.38, 0.37]]
print("inner score table:", t4_inner_scores.tolist())  # -> [[0.44, 0.36, 0.39], [0.31, 0.34, 0.33], [0.41, 0.38, 0.37]]
t4_chosen_cols = np.argmin(t4_inner_scores, axis=1)  # best column per outer fold  # -> [1, 0, 2]
print("chosen columns:", t4_chosen_cols.tolist())  # -> [1, 0, 2]
t4_chosen_lambdas = t4_lambdas[t4_chosen_cols]  # selected lambda per outer fold  # -> [0.1, 0.01, 1.0]
print("chosen lambdas:", t4_chosen_lambdas.tolist())  # -> [0.1, 0.01, 1.0]
assert np.allclose(t4_chosen_lambdas, [0.10, 0.01, 1.00])

plt.figure(figsize=(4.6, 3.0))
plt.imshow(t4_inner_scores, cmap="viridis")
plt.colorbar(label="inner score")
plt.xticks(np.arange(3), ["0.01", "0.10", "1.00"])
plt.yticks(np.arange(3), ["outer 1", "outer 2", "outer 3"])
plt.title("Toy 4 · choose the row minimum")
plt.show()

▶ What you'll see: the selected λ can differ by outer fold because each row has a different minimum.

### ✍️ Toy 5 · Outer scores average into the nested estimate

After inner tuning, the untouched outer test scores are averaged to estimate the full tuning workflow.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)  # seed the toy for reproducibility.
t5_outer_scores = np.array([0.32, 0.36, 0.34, 0.31, 0.35, 0.33])  # held-out scores after tuning  # -> [0.32, 0.36, 0.34, 0.31, 0.35, 0.33]
print("outer held-out scores:", t5_outer_scores.tolist())  # -> [0.32, 0.36, 0.34, 0.31, 0.35, 0.33]
t5_outer_sum = float(t5_outer_scores.sum())  # total held-out loss across outer folds  # -> 2.01
print("outer score sum:", round(t5_outer_sum, 3))  # -> 2.01
t5_outer_count = int(t5_outer_scores.size)  # number of outer folds  # -> 6
print("number of outer folds:", t5_outer_count)  # -> 6
t5_nested_estimate = float(t5_outer_scores.mean())  # nested CV estimate  # -> 0.335
print("nested CV estimate:", round(t5_nested_estimate, 3))  # -> 0.335
assert round(t5_nested_estimate, 3) == 0.335

plt.figure(figsize=(4.8, 2.8))
plt.plot(np.arange(1, t5_outer_count + 1), t5_outer_scores, marker="o", color="purple")
plt.axhline(t5_nested_estimate, color="black", linestyle="--", label="mean = 0.335")
plt.xlabel("outer fold")
plt.ylabel("held-out loss")
plt.title("Toy 5 · average outer test folds")
plt.legend()
plt.show()

▶ What you'll see: six held-out losses wobble around their mean, the nested estimate `0.335`.

### ✍️ Toy 6 · Reusing the tuning table is optimistic

A non-nested report takes the minimum validation score from the same table it searched, which tends to be too low.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)  # seed the toy for reproducibility.
t6_flat_scores = np.array([0.35, 0.31, 0.33, 0.34, 0.30, 0.32])  # scores searched and reported by flat CV  # -> [0.35, 0.31, 0.33, 0.34, 0.3, 0.32]
print("flat CV candidate scores:", t6_flat_scores.tolist())  # -> [0.35, 0.31, 0.33, 0.34, 0.3, 0.32]
t6_selected = int(np.argmin(t6_flat_scores))  # selected candidate index  # -> 4
print("selected candidate index:", t6_selected)  # -> 4
t6_reported = float(t6_flat_scores[t6_selected])  # reused minimum score  # -> 0.3
print("non-nested reported score:", round(t6_reported, 3))  # -> 0.3
t6_honest_scores = np.array([0.33, 0.34, 0.32, 0.36, 0.35, 0.31])  # untouched outer-style scores  # -> [0.33, 0.34, 0.32, 0.36, 0.35, 0.31]
print("honest held-out scores:", t6_honest_scores.tolist())  # -> [0.33, 0.34, 0.32, 0.36, 0.35, 0.31]
t6_honest_estimate = float(t6_honest_scores.mean())  # held-out estimate  # -> 0.335
print("honest estimate:", round(t6_honest_estimate, 3))  # -> 0.335
t6_optimism = t6_honest_estimate - t6_reported  # optimism gap from reuse  # -> 0.035
print("optimism gap:", round(t6_optimism, 3))  # -> 0.035
assert round(t6_optimism, 3) == 0.035

plt.figure(figsize=(4.8, 2.8))
plt.bar(["reused min", "held-out mean"], [t6_reported, t6_honest_estimate], color=["crimson", "seagreen"])
plt.ylabel("estimated loss")
plt.title("Toy 6 · the reused minimum is lower")
plt.show()

▶ What you'll see: the non-nested report is lower by `0.035`, but that low number came from reuse.

### ✍️ Toy 7 · Stabilization wins only after the final argmin

The final decision compares complete scores after costs and stabilization factors have been applied.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)  # seed the toy for reproducibility.
t7_labels = np.array(["baseline", "flexible", "stabilized"])  # candidate names  # -> ['baseline', 'flexible', 'stabilized']
print("candidate labels:", t7_labels.tolist())  # -> ['baseline', 'flexible', 'stabilized']
t7_raw_risks = np.array([0.28, 0.26, 0.28])  # raw validation risks  # -> [0.28, 0.26, 0.28]
print("raw risks:", t7_raw_risks.tolist())  # -> [0.28, 0.26, 0.28]
t7_costs = np.array([0.06, 0.10, 0.02])  # candidate costs  # -> [0.06, 0.1, 0.02]
print("costs:", t7_costs.tolist())  # -> [0.06, 0.1, 0.02]
t7_scores = t7_raw_risks + t7_costs  # complete decision scores before stabilization  # -> [0.34, 0.36, 0.3]
print("risk plus cost:", np.round(t7_scores, 3).tolist())  # -> [0.34, 0.36, 0.3]
t7_multipliers = np.array([1.00, 1.00, 0.85])  # stabilization adjustment  # -> [1.0, 1.0, 0.85]
print("stability multipliers:", t7_multipliers.tolist())  # -> [1.0, 1.0, 0.85]
t7_final_scores = t7_scores * t7_multipliers  # final scores to minimize  # -> [0.34, 0.36, 0.255]
print("final scores:", np.round(t7_final_scores, 3).tolist())  # -> [0.34, 0.36, 0.255]
t7_winner = int(np.argmin(t7_final_scores))  # lower final score wins  # -> 2
print("winner:", str(t7_labels[t7_winner]))  # -> stabilized
assert str(t7_labels[t7_winner]) == "stabilized"

plt.figure(figsize=(5.0, 2.8))
plt.bar(t7_labels, t7_final_scores, color=["gray", "orange", "seagreen"])
plt.ylabel("final decision score")
plt.title("Toy 7 · minimize after stabilization")
plt.show()

▶ What you'll see: the stabilized candidate wins because its final score is `0.255`.


## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, folds, losses, and small deterministic simulations.
import matplotlib.pyplot as plt # load Matplotlib for fold diagrams, curves, and score comparisons.
np.random.seed(0) # make all examples reproducible across runs.

## 🟢 Basics (warm-up)

### Basic 1 — Average three losses

**Goal.** Compute empirical risk from per-example losses, because every cross-validation fold score is an average loss. We build it in 2 steps.

In [ ]:
losses_b1 = np.array([0.191, 0.135, 0.454]) # store the lesson's three verified losses.
print("losses_b1:", losses_b1) # inspect the per-example values before reducing them.
print("count:", len(losses_b1)) # inspect the denominator of the average.

▶ What you'll see: three losses and a denominator of 3.

In [ ]:
risk_b1 = float(np.mean(losses_b1)) # average losses to estimate expected loss per example.
print("R_S:", round(risk_b1, 3)) # inspect the empirical risk.
assert round(risk_b1, 3) == 0.260 # verify the lesson arithmetic.
plt.figure(figsize=(4, 3)) # create a compact loss plot.
plt.bar(["1", "2", "3"], losses_b1, color="steelblue") # show each example's contribution.
plt.axhline(risk_b1, color="red", linestyle="--") # show the average loss.
plt.title("Basic 1: average loss") # title the plot.
plt.ylabel("loss") # label the loss axis.
plt.show() # display the plot.

▶ What you'll see: the mean line summarizes the three losses at `0.260`.

👀 Takeaway: empirical risk is a mean, so fold scores remain comparable across fold sizes.

### Basic 2 — Add a cost term

**Goal.** Convert raw risk into a selection score, because model choice may need to include complexity, regularization, or operational cost. We build it in 2 steps.

In [ ]:
risk_b2 = 0.260 # use the verified raw empirical risk.
cost_b2 = 0.060 # use the lesson's method cost.
print("raw risk:", risk_b2) # inspect the first score component.
print("cost:", cost_b2) # inspect the second score component.

▶ What you'll see: the two ingredients that must be added before selection.

In [ ]:
score_b2 = risk_b2 + cost_b2 # compute the decision score.
print("score:", round(score_b2, 3)) # inspect the full selection score.
assert round(score_b2, 3) == 0.320 # verify 0.260 + 0.060.
plt.figure(figsize=(4, 3)) # create a compact component plot.
plt.bar(["risk", "cost", "score"], [risk_b2, cost_b2, score_b2], color=["steelblue", "orange", "green"]) # compare pieces and total.
plt.title("Basic 2: score includes cost") # title the plot.
plt.show() # display the plot.

▶ What you'll see: the score bar is larger than the raw-risk bar because cost is included.

👀 Takeaway: selection should optimize the full score implied by the method, not only the raw fit.

### Basic 3 — Compare two candidate scores

**Goal.** Rank two hyperparameter settings by their validation score, because the inner loop chooses the lowest validation loss. We build it in 2 steps.

In [ ]:
scores_b3 = np.array([0.320, 0.364]) # compare baseline and flexible candidate scores.
names_b3 = np.array(["baseline", "flexible"]) # name the candidates for readable output.
print("candidate scores:", dict(zip(names_b3, scores_b3))) # inspect the score table.

▶ What you'll see: a two-row decision table with lower-is-better scores.

In [ ]:
winner_b3 = int(np.argmin(scores_b3)) # choose the candidate with the smaller score.
print("winner:", names_b3[winner_b3]) # inspect the selected candidate.
assert names_b3[winner_b3] == "baseline" # verify the lower score wins.
plt.figure(figsize=(4, 3)) # create a candidate comparison plot.
plt.bar(names_b3, scores_b3, color=["seagreen", "gray"]) # visualize the lower score.
plt.title("Basic 3: lower validation score wins") # title the plot.
plt.ylabel("score") # label the score axis.
plt.show() # display the plot.

▶ What you'll see: the baseline bar is shorter, so it is selected.

👀 Takeaway: validation selection is an argmin over candidate scores.

### Basic 4 — Compute the absolute and relative gap

**Goal.** Measure the size of a win, because a lower score is more convincing when the gap is large relative to the scale. We build it in 2 steps.

In [ ]:
best_b4 = 0.320 # score for the better candidate.
other_b4 = 0.364 # score for the tempting alternative.
gap_b4 = other_b4 - best_b4 # absolute improvement in loss units.
print("absolute gap:", round(gap_b4, 3)) # inspect the raw difference.
assert round(gap_b4, 3) == 0.044 # verify the lesson gap.

▶ What you'll see: the baseline wins by `0.044` loss units.

In [ ]:
relative_gap_b4 = gap_b4 / other_b4 # scale the gap by the alternative score.
print("relative gap:", round(relative_gap_b4, 3)) # inspect the scale-free difference.
assert round(relative_gap_b4, 3) == 0.121 # verify the lesson relative gap.
plt.figure(figsize=(4, 3)) # create a small gap plot.
plt.bar(["gap", "relative gap"], [gap_b4, relative_gap_b4], color=["purple", "orange"]) # compare raw and relative evidence.
plt.title("Basic 4: size of the win") # title the plot.
plt.show() # display the plot.

▶ What you'll see: the relative gap gives the same win on a dimensionless scale.

👀 Takeaway: the gap is the evidence margin that must survive resampling noise.

### Basic 5 — Split indices into folds

**Goal.** Create simple fold assignments, because cross-validation repeatedly holds out different subsets. We build it in 2 steps.

In [ ]:
n_b5 = 12 # create twelve example indices.
K_b5 = 3 # choose three folds.
indices_b5 = np.arange(n_b5) # examples are represented by integer positions.
print("indices:", indices_b5) # inspect all examples before splitting.

▶ What you'll see: examples `0` through `11` ready to be assigned to folds.

In [ ]:
fold_ids_b5 = indices_b5 % K_b5 # assign examples cyclically to folds 0, 1, 2.
fold_sizes_b5 = np.array([np.sum(fold_ids_b5 == k) for k in range(K_b5)]) # count examples per fold.
print("fold ids:", fold_ids_b5) # inspect each example's fold.
print("fold sizes:", fold_sizes_b5) # inspect balance.
assert np.all(fold_sizes_b5 == 4) # verify all folds are equal here.
plt.figure(figsize=(5, 2.6)) # create a fold assignment plot.
plt.scatter(indices_b5, fold_ids_b5, c=fold_ids_b5, cmap="viridis", s=80) # show example-to-fold assignment.
plt.yticks([0, 1, 2]) # show fold labels.
plt.title("Basic 5: fold assignment") # title the plot.
plt.xlabel("example index") # label examples.
plt.ylabel("fold") # label folds.
plt.show() # display the plot.

▶ What you'll see: four examples assigned to each of three folds.

👀 Takeaway: CV works by rotating which fold is held out for validation or testing.

### Basic 6 — Make one train/test mask

**Goal.** Build the masks for one outer fold, because nested CV must keep the outer test fold untouched. We build it in 2 steps.

In [ ]:
fold_ids_b6 = np.arange(12) % 3 # reuse a balanced three-fold assignment.
outer_fold_b6 = 1 # choose fold 1 as the held-out outer test fold.
test_mask_b6 = fold_ids_b6 == outer_fold_b6 # mark examples excluded from model selection.
train_mask_b6 = ~test_mask_b6 # the remaining examples are available for inner CV.
print("test indices:", np.where(test_mask_b6)[0]) # inspect held-out examples.

▶ What you'll see: examples in fold 1 are held out.

In [ ]:
print("train count:", int(train_mask_b6.sum()), "test count:", int(test_mask_b6.sum())) # inspect the split sizes.
assert int(train_mask_b6.sum()) == 8 and int(test_mask_b6.sum()) == 4 # verify 8/4 split.
plt.figure(figsize=(5, 2.6)) # create a split visualization.
plt.bar(np.arange(12), train_mask_b6.astype(int), label="train", color="seagreen") # show training membership.
plt.bar(np.arange(12), test_mask_b6.astype(int), bottom=train_mask_b6.astype(int), label="outer test", color="crimson") # show test membership.
plt.title("Basic 6: one outer split") # title the plot.
plt.xlabel("example") # label examples.
plt.legend() # show color meanings.
plt.show() # display the plot.

▶ What you'll see: exactly four examples are reserved as the outer test fold.

👀 Takeaway: the outer test fold estimates performance and must not tune hyperparameters.

### Basic 7 — Select λ inside one fold

**Goal.** Choose a hyperparameter from inner validation scores, because the inner loop owns model selection. We build it in 2 steps.

In [ ]:
lambdas_b7 = np.array([0.0, 0.1, 1.0]) # candidate regularization strengths.
inner_scores_b7 = np.array([0.340, 0.300, 0.315]) # validation losses for one outer fold's inner loop.
print("λ grid:", lambdas_b7) # inspect candidates.
print("inner scores:", inner_scores_b7) # inspect validation estimates.

▶ What you'll see: λ=0.1 has the lowest inner validation loss.

In [ ]:
best_idx_b7 = int(np.argmin(inner_scores_b7)) # choose the lowest inner validation score.
best_lambda_b7 = float(lambdas_b7[best_idx_b7]) # read the selected λ.
print("best λ:", best_lambda_b7) # inspect the selected setting.
assert best_lambda_b7 == 0.1 # verify the intended inner-loop choice.
plt.figure(figsize=(4, 3)) # create a hyperparameter sweep plot.
plt.plot(lambdas_b7, inner_scores_b7, marker="o", color="navy") # plot validation loss by λ.
plt.axvline(best_lambda_b7, color="red", linestyle="--") # mark the winner.
plt.title("Basic 7: inner-loop selection") # title the plot.
plt.xlabel("λ") # label candidate axis.
plt.ylabel("validation loss") # label loss axis.
plt.show() # display the plot.

▶ What you'll see: the dashed line marks the validation-minimizing λ.

👀 Takeaway: the inner loop returns a selected hyperparameter, not the final performance estimate.

### Basic 8 — Average outer scores

**Goal.** Combine outer held-out losses, because nested CV reports the average performance of the tuned procedure. We build it in 2 steps.

In [ ]:
outer_scores_b8 = np.array([0.322, 0.351, 0.305]) # held-out losses from three outer folds.
print("outer scores:", outer_scores_b8) # inspect fold-by-fold test performance.
print("outer count:", len(outer_scores_b8)) # inspect the denominator for the estimate.

▶ What you'll see: three held-out scores, one per outer fold.

In [ ]:
nested_b8 = float(np.mean(outer_scores_b8)) # average the outer losses.
print("nested estimate:", round(nested_b8, 3)) # inspect the final estimate.
assert round(nested_b8, 3) == 0.326 # verify the walkthrough value.
plt.figure(figsize=(4, 3)) # create an outer-score plot.
plt.bar(["fold1", "fold2", "fold3"], outer_scores_b8, color="purple") # show each held-out score.
plt.axhline(nested_b8, color="black", linestyle="--") # show the average.
plt.title("Basic 8: average outer folds") # title the plot.
plt.ylabel("outer loss") # label the loss axis.
plt.show() # display the plot.

▶ What you'll see: the dashed line is the nested CV estimate.

👀 Takeaway: the reported nested score is an outer-fold average, not an inner-fold minimum.

### Basic 9 — Show non-nested optimism

**Goal.** Compare a reused validation minimum with the nested estimate, because non-nested CV can be too optimistic. We build it in 2 steps.

In [ ]:
non_nested_b9 = 0.301 # minimum from a flat CV table reused for reporting.
nested_b9 = 0.326 # honest outer-fold estimate of the selection procedure.
optimism_b9 = nested_b9 - non_nested_b9 # positive means the flat report looks too good.
print("non-nested report:", non_nested_b9) # inspect the optimistic score.
print("nested estimate:", nested_b9) # inspect the honest score.

▶ What you'll see: the non-nested number is smaller.

In [ ]:
print("optimism gap:", round(optimism_b9, 3)) # inspect the bias size.
assert round(optimism_b9, 3) == 0.025 # verify the optimism gap.
plt.figure(figsize=(4, 3)) # create a comparison chart.
plt.bar(["non-nested", "nested"], [non_nested_b9, nested_b9], color=["red", "green"]) # compare estimates.
plt.title("Basic 9: optimistic reuse") # title the plot.
plt.ylabel("estimated loss") # label loss axis.
plt.show() # display the plot.

▶ What you'll see: the reused validation estimate appears better by `0.025`.

👀 Takeaway: validation used for selection should not also be treated as a clean test estimate.

### Basic 10 — Pick the stabilized option

**Goal.** Make the final three-way decision, because the lesson compares baseline, flexible, and stabilized scores. We build it in 2 steps.

In [ ]:
scores_b10 = np.array([0.320, 0.364, 0.256]) # baseline, flexible, and stabilized scores.
labels_b10 = np.array(["baseline", "flexible", "stabilized"]) # candidate names.
print("scores:", dict(zip(labels_b10, scores_b10))) # inspect the final decision table.

▶ What you'll see: the stabilized option has the smallest score.

In [ ]:
winner_b10 = int(np.argmin(scores_b10)) # choose the lowest score.
print("winner:", labels_b10[winner_b10]) # inspect the selected option.
assert labels_b10[winner_b10] == "stabilized" # verify the lesson decision.
plt.figure(figsize=(5, 3)) # create a final decision plot.
plt.bar(labels_b10, scores_b10, color=["gray", "orange", "seagreen"]) # compare all candidates.
plt.title("Basic 10: final minimum score") # title the plot.
plt.ylabel("lower is better") # label score axis.
plt.show() # display the plot.

▶ What you'll see: the stabilized bar is shortest.

👀 Takeaway: choose using the complete score table, not the most attractive training fragment.

## 🟡 Easy

### Easy 1 — Run a full nested-CV bookkeeping table

**Goal.** Simulate the bookkeeping of nested CV, because each outer fold must select inside and evaluate outside. We build it in 3 steps.

In [ ]:
lambdas_e1 = np.array([0.0, 0.1, 1.0]) # candidate hyperparameters.
inner_e1 = np.array([[0.340, 0.300, 0.315], [0.330, 0.318, 0.309], [0.355, 0.325, 0.333]]) # inner losses by outer fold.
outer_by_candidate_e1 = np.array([[0.350, 0.322, 0.340], [0.360, 0.358, 0.351], [0.330, 0.305, 0.320]]) # outer losses if each candidate were chosen.
print("inner table shape:", inner_e1.shape) # inspect rows and candidates.

▶ What you'll see: a 3 outer-fold by 3 candidate inner-score table.

In [ ]:
chosen_e1 = np.argmin(inner_e1, axis=1) # select one candidate per outer fold using only inner scores.
chosen_lambdas_e1 = lambdas_e1[chosen_e1] # convert indices into λ values.
outer_scores_e1 = outer_by_candidate_e1[np.arange(3), chosen_e1] # evaluate selected candidates on outer folds.
print("chosen λ:", chosen_lambdas_e1) # inspect selections.
print("outer scores:", outer_scores_e1) # inspect held-out outcomes.
assert np.allclose(chosen_lambdas_e1, [0.1, 1.0, 0.1]) # verify selections.

In [ ]:
nested_e1 = float(np.mean(outer_scores_e1)) # average outer scores.
print("nested estimate:", round(nested_e1, 3)) # inspect final estimate.
assert round(nested_e1, 3) == 0.326 # verify the expected nested estimate.
plt.figure(figsize=(5, 3)) # create a fold-level plot.
plt.bar(["fold1", "fold2", "fold3"], outer_scores_e1, color="purple") # show selected outer scores.
plt.axhline(nested_e1, color="black", linestyle="--") # show average.
plt.title("Easy 1: selected outer scores") # title the plot.
plt.ylabel("outer loss") # label loss axis.
plt.show() # display the plot.

▶ What you'll see: the nested estimate averages only the outer losses of settings selected by the inner loop.

👀 Takeaway: nested CV evaluates the procedure "select inside, score outside" fold by fold.

### Easy 2 — Compare nested and flat selection on noisy candidates

**Goal.** Show why selecting the minimum from one noisy table is optimistic, because the lowest validation estimate may be lucky. We build it in 3 steps.

In [ ]:
true_losses_e2 = np.array([0.330, 0.325, 0.335, 0.345]) # pretend these are true future losses.
noise_e2 = np.array([0.004, -0.024, -0.018, 0.002]) # validation noise on one flat CV run.
flat_scores_e2 = true_losses_e2 + noise_e2 # observed validation scores.
print("flat validation scores:", np.round(flat_scores_e2, 3)) # inspect noisy estimates.

▶ What you'll see: the second candidate looks best partly because its noise is negative.

In [ ]:
chosen_e2 = int(np.argmin(flat_scores_e2)) # choose by the noisy flat table.
reported_e2 = float(flat_scores_e2[chosen_e2]) # non-nested report reuses the lucky value.
future_e2 = float(true_losses_e2[chosen_e2]) # actual future-like loss for that candidate.
print("chosen candidate:", chosen_e2) # inspect selected index.
print("reported vs future:", round(reported_e2, 3), round(future_e2, 3)) # compare optimistic and true values.
assert round(reported_e2, 3) == 0.301 and round(future_e2, 3) == 0.325 # verify the optimism example.

In [ ]:
optimism_e2 = future_e2 - reported_e2 # optimistic amount from reusing noisy validation.
print("optimism:", round(optimism_e2, 3)) # inspect bias size.
assert round(optimism_e2, 3) == 0.024 # verify the gap.
plt.figure(figsize=(4, 3)) # create a comparison plot.
plt.bar(["reported", "future-like"], [reported_e2, future_e2], color=["red", "green"]) # visualize optimistic bias.
plt.title("Easy 2: flat CV optimism") # title the plot.
plt.ylabel("loss") # label loss axis.
plt.show() # display the plot.

▶ What you'll see: the reused validation score is lower than the future-like loss.

👀 Takeaway: choosing the minimum of noisy validation estimates makes the reported minimum biased downward.

### Easy 3 — Build inner folds only from outer training data

**Goal.** Verify that inner validation folds never include outer-test examples, because leakage breaks the nested-CV contract. We build it in 3 steps.

In [ ]:
n_e3 = 15 # use fifteen examples for a small split demo.
outer_fold_ids_e3 = np.arange(n_e3) % 3 # assign examples to three outer folds.
outer_test_e3 = outer_fold_ids_e3 == 2 # hold out outer fold 2.
outer_train_indices_e3 = np.where(~outer_test_e3)[0] # only these may enter inner CV.
print("outer train indices:", outer_train_indices_e3) # inspect allowed examples.
print("outer test indices:", np.where(outer_test_e3)[0]) # inspect forbidden examples.

▶ What you'll see: five examples are forbidden outer-test examples.

In [ ]:
inner_fold_ids_e3 = np.arange(len(outer_train_indices_e3)) % 2 # split only outer-training examples into two inner folds.
inner_val_indices_e3 = outer_train_indices_e3[inner_fold_ids_e3 == 1] # map inner validation positions back to original indices.
leak_count_e3 = int(np.sum(outer_test_e3[inner_val_indices_e3])) # count any forbidden overlap.
print("inner validation indices:", inner_val_indices_e3) # inspect inner validation examples.
print("leak count:", leak_count_e3) # verify no outer-test data leaked in.
assert leak_count_e3 == 0 # inner folds are contained within outer training.

In [ ]:
plt.figure(figsize=(5, 2.8)) # create a containment plot.
plt.scatter(outer_train_indices_e3, np.zeros_like(outer_train_indices_e3), label="outer train", color="seagreen", s=80) # show outer-train examples.
plt.scatter(np.where(outer_test_e3)[0], np.ones(np.sum(outer_test_e3)), label="outer test", color="crimson", s=80) # show outer-test examples.
plt.scatter(inner_val_indices_e3, np.full_like(inner_val_indices_e3, 0.4), label="inner val", color="navy", s=45) # show inner val inside outer train.
plt.yticks([0, 0.4, 1], ["outer train", "inner val", "outer test"]) # label rows.
plt.title("Easy 3: inner folds stay inside outer train") # title the plot.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: inner validation points sit within the outer-training row, not the outer-test row.

👀 Takeaway: nested CV prevents leakage by constructing inner folds after removing the outer test fold.

### Easy 4 — Tune a polynomial degree with manual validation loss

**Goal.** Select model flexibility with a tiny from-scratch regression example, because nested CV can tune any hyperparameter. We build it in 4 steps.

In [ ]:
x_e4 = np.linspace(-1, 1, 12) # create one-dimensional inputs.
y_e4 = 1.0 + 2.0 * x_e4 - 0.8 * x_e4 ** 2 + np.array([0.05, -0.02, 0.03, -0.04, 0.02, 0.00, -0.01, 0.04, -0.03, 0.02, -0.02, 0.01]) # deterministic noisy quadratic targets.
val_e4 = np.array([2, 5, 8, 11]) # hold out four spread-out validation examples.
train_e4 = np.setdiff1d(np.arange(12), val_e4) # use the remaining eight examples as training data.
print("train size:", len(train_e4), "val size:", len(val_e4)) # inspect split sizes.

▶ What you'll see: a small 8/4 split with validation points spread across the input range.

In [ ]:
degrees_e4 = np.array([1, 2, 3]) # candidate polynomial degrees.
val_losses_e4 = [] # store validation MSE per degree.
for degree_e4 in degrees_e4: # fit each candidate.
    X_train_e4 = np.vander(x_e4[train_e4], N=degree_e4 + 1, increasing=True) # polynomial design matrix for training.
    X_val_e4 = np.vander(x_e4[val_e4], N=degree_e4 + 1, increasing=True) # polynomial design matrix for validation.
    coef_e4 = np.linalg.pinv(X_train_e4) @ y_e4[train_e4] # least-squares fit from scratch.
    pred_val_e4 = X_val_e4 @ coef_e4 # validation predictions.
    val_losses_e4.append(float(np.mean((y_e4[val_e4] - pred_val_e4) ** 2))) # validation MSE.
print("validation losses:", np.round(val_losses_e4, 4)) # inspect selection scores.

In [ ]:
best_degree_e4 = int(degrees_e4[int(np.argmin(val_losses_e4))]) # select degree with lowest validation loss.
print("best degree:", best_degree_e4) # inspect selected flexibility.
assert best_degree_e4 == 2 # the data were generated by a quadratic pattern.

In [ ]:
plt.figure(figsize=(5, 3)) # create a degree-sweep plot.
plt.plot(degrees_e4, val_losses_e4, marker="o", color="purple") # plot validation loss by degree.
plt.axvline(best_degree_e4, color="red", linestyle="--") # mark selected degree.
plt.title("Easy 4: inner validation chooses degree") # title the plot.
plt.xlabel("polynomial degree") # label degree axis.
plt.ylabel("validation MSE") # label loss axis.
plt.show() # display the plot.

▶ What you'll see: degree 2 has the smallest validation MSE on this toy split.

👀 Takeaway: inner CV can tune model flexibility without using the outer test fold.

### Easy 5 — Estimate uncertainty across outer folds

**Goal.** Summarize outer-fold variability, because a nested-CV mean should be read with its fold-to-fold uncertainty. We build it in 3 steps.

In [ ]:
outer_scores_e5 = np.array([0.322, 0.351, 0.305, 0.334, 0.318]) # five held-out scores from a nested run.
mean_e5 = float(np.mean(outer_scores_e5)) # average outer performance.
std_e5 = float(np.std(outer_scores_e5, ddof=1)) # sample standard deviation across folds.
print("outer scores:", outer_scores_e5) # inspect fold losses.
print("mean:", round(mean_e5, 3), "std:", round(std_e5, 3)) # inspect summary statistics.

▶ What you'll see: outer scores vary around a mean near `0.326`.

In [ ]:
se_e5 = std_e5 / np.sqrt(len(outer_scores_e5)) # standard error of the mean across folds.
lo_e5 = mean_e5 - 2 * se_e5 # rough lower band.
hi_e5 = mean_e5 + 2 * se_e5 # rough upper band.
print("rough ±2SE interval:", round(lo_e5, 3), "to", round(hi_e5, 3)) # inspect uncertainty band.
assert round(mean_e5, 3) == 0.326 # verify the mean.

In [ ]:
plt.figure(figsize=(5, 3)) # create an uncertainty visualization.
plt.errorbar([0], [mean_e5], yerr=[2 * se_e5], fmt="o", color="black", capsize=6, label="mean ± 2SE") # plot summary uncertainty.
plt.scatter(np.zeros_like(outer_scores_e5) + 0.08, outer_scores_e5, color="purple", label="folds") # show individual folds.
plt.xlim(-0.5, 0.7) # keep points visible.
plt.xticks([0], ["nested CV"]) # label x-axis.
plt.ylabel("outer loss") # label loss axis.
plt.title("Easy 5: outer-fold uncertainty") # title the plot.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: individual fold scores surround a mean with a rough uncertainty bar.

👀 Takeaway: the outer-fold mean is an estimate, so fold variability matters when gaps are small.

## 🔴 Advanced

### Advanced 1 — Manual nested CV for ridge regression

**Goal.** Implement nested CV end to end with only NumPy, because the method is a protocol rather than a library call. We build it in 5 steps.

In [ ]:
rng_a1 = np.random.default_rng(1) # create a local generator for reproducible data.
x_a1 = np.linspace(-2, 2, 30) # one-dimensional inputs.
y_a1 = 1.0 + 1.5 * x_a1 - 0.7 * x_a1 ** 2 + 0.15 * rng_a1.normal(size=x_a1.size) # noisy quadratic target.
outer_ids_a1 = np.arange(x_a1.size) % 3 # deterministic three outer folds.
lams_a1 = np.array([0.0, 0.01, 0.1, 1.0]) # ridge penalty candidates.
print("data size:", x_a1.size, "lambda grid:", lams_a1) # inspect experiment setup.

▶ What you'll see: 30 examples, 3 outer folds, and 4 ridge penalties.

In [ ]:
outer_scores_a1 = [] # store held-out MSE per outer fold.
chosen_lams_a1 = [] # store selected λ per outer fold.
for outer_a1 in range(3): # loop over outer folds.
    outer_test_a1 = outer_ids_a1 == outer_a1 # held-out fold for final evaluation.
    outer_train_a1 = ~outer_test_a1 # data available for inner selection.
    train_idx_a1 = np.where(outer_train_a1)[0] # original indices allowed in inner CV.
    inner_ids_a1 = np.arange(len(train_idx_a1)) % 2 # two inner folds inside outer training.
    inner_means_a1 = [] # validation loss per lambda.
    for lam_a1 in lams_a1: # evaluate each candidate in inner CV.
        fold_losses_a1 = [] # collect inner validation losses.
        for inner_a1 in range(2): # loop over inner validation folds.
            val_idx_a1 = train_idx_a1[inner_ids_a1 == inner_a1] # inner validation indices.
            fit_idx_a1 = train_idx_a1[inner_ids_a1 != inner_a1] # inner training indices.
            X_fit_a1 = np.vander(x_a1[fit_idx_a1], N=3, increasing=True) # quadratic design.
            X_val_a1 = np.vander(x_a1[val_idx_a1], N=3, increasing=True) # validation design.
            reg_a1 = lam_a1 * np.eye(3) # ridge penalty matrix.
            coef_a1 = np.linalg.solve(X_fit_a1.T @ X_fit_a1 + reg_a1, X_fit_a1.T @ y_a1[fit_idx_a1]) # ridge fit.
            fold_losses_a1.append(float(np.mean((y_a1[val_idx_a1] - X_val_a1 @ coef_a1) ** 2))) # inner MSE.
        inner_means_a1.append(float(np.mean(fold_losses_a1))) # mean inner CV loss for λ.
    best_a1 = int(np.argmin(inner_means_a1)) # choose λ inside this outer fold.
    chosen_lams_a1.append(float(lams_a1[best_a1])) # record selected λ.
    X_train_a1 = np.vander(x_a1[outer_train_a1], N=3, increasing=True) # refit design on full outer training.
    X_test_a1 = np.vander(x_a1[outer_test_a1], N=3, increasing=True) # outer test design.
    reg_best_a1 = lams_a1[best_a1] * np.eye(3) # selected ridge penalty.
    coef_best_a1 = np.linalg.solve(X_train_a1.T @ X_train_a1 + reg_best_a1, X_train_a1.T @ y_a1[outer_train_a1]) # refit selected model.
    outer_scores_a1.append(float(np.mean((y_a1[outer_test_a1] - X_test_a1 @ coef_best_a1) ** 2))) # score on untouched outer fold.
print("chosen λ:", chosen_lams_a1) # inspect fold-specific choices.
print("outer MSE:", np.round(outer_scores_a1, 4)) # inspect held-out scores.

In [ ]:
nested_mse_a1 = float(np.mean(outer_scores_a1)) # final nested estimate.
print("nested MSE:", round(nested_mse_a1, 4)) # inspect estimate.
assert nested_mse_a1 < 0.08 # sanity-check this simple quadratic problem is fit well.

In [ ]:
plt.figure(figsize=(5, 3)) # create an outer-score plot.
plt.plot(np.arange(1, 4), outer_scores_a1, marker="o", color="teal") # show held-out MSE per outer fold.
plt.axhline(nested_mse_a1, color="black", linestyle="--", label=f"mean={nested_mse_a1:.3f}") # show mean.
plt.title("Advanced 1: manual nested ridge CV") # title the plot.
plt.xlabel("outer fold") # label outer fold axis.
plt.ylabel("outer MSE") # label MSE axis.
plt.xticks([1, 2, 3]) # show fold numbers.
plt.legend() # show mean label.
plt.show() # display the plot.

▶ What you'll see: each outer fold selects λ internally, then contributes one untouched MSE to the final estimate.

👀 Takeaway: nested CV is just careful indexing: tune inside outer-train data, then score once on outer-test data.

### Advanced 2 — Show leakage by tuning on the outer fold

**Goal.** Compare a correct nested estimate with a leaky estimate, because using outer-test data during selection makes performance look too good. We build it in 4 steps.

In [ ]:
outer_truth_a2 = np.array([0.340, 0.330, 0.335]) # true-ish outer losses for three candidates.
outer_noise_a2 = np.array([0.010, -0.030, -0.005]) # noise observed on one outer fold.
outer_observed_a2 = outer_truth_a2 + outer_noise_a2 # leaky procedure sees these while selecting.
inner_observed_a2 = np.array([0.342, 0.336, 0.337]) # correct inner validation scores.
print("outer observed:", np.round(outer_observed_a2, 3)) # inspect forbidden selection data.
print("inner observed:", inner_observed_a2) # inspect allowed selection data.

▶ What you'll see: candidate 1 looks especially good on the outer fold because of negative noise.

In [ ]:
leaky_choice_a2 = int(np.argmin(outer_observed_a2)) # wrong: choose by outer-test score.
proper_choice_a2 = int(np.argmin(inner_observed_a2)) # right: choose by inner validation score.
leaky_report_a2 = float(outer_observed_a2[leaky_choice_a2]) # report the outer score after selecting on it.
proper_outer_a2 = float(outer_truth_a2[proper_choice_a2]) # evaluate inner-selected candidate on future-like truth.
print("leaky choice:", leaky_choice_a2, "proper choice:", proper_choice_a2) # inspect choices.
print("leaky report:", round(leaky_report_a2, 3), "proper outer:", round(proper_outer_a2, 3)) # compare estimates.

In [ ]:
leak_bias_a2 = proper_outer_a2 - leaky_report_a2 # optimistic gap caused by leakage.
print("leakage optimism:", round(leak_bias_a2, 3)) # inspect magnitude.
assert round(leaky_report_a2, 3) == 0.300 # verify the flattering leaky report.
assert round(leak_bias_a2, 3) == 0.030 # verify the bias in this toy case.

In [ ]:
plt.figure(figsize=(4.8, 3)) # create a leakage comparison plot.
plt.bar(["leaky report", "proper outer"], [leaky_report_a2, proper_outer_a2], color=["crimson", "seagreen"]) # show optimistic versus proper estimate.
plt.title("Advanced 2: outer-fold leakage") # title the plot.
plt.ylabel("loss") # label loss axis.
plt.show() # display the plot.

▶ What you'll see: the leaky report is lower because it selected on the same fold it reported.

👀 Takeaway: once a fold influences selection, it is no longer an honest test fold.

### Advanced 3 — Repeated nested CV reduces split noise

**Goal.** Repeat a nested-CV-like outer-score simulation, because one split can be noisy and repeated estimates expose variability. We build it in 4 steps.

In [ ]:
base_outer_a3 = np.array([0.322, 0.351, 0.305]) # base outer losses from the walkthrough.
repeat_offsets_a3 = np.array([[0.000, 0.000, 0.000], [0.006, -0.004, 0.003], [-0.005, 0.002, -0.001], [0.003, 0.005, -0.004]]) # deterministic repeat-to-repeat perturbations.
repeated_scores_a3 = base_outer_a3 + repeat_offsets_a3 # four repeated nested-CV runs.
print("repeated outer scores:\n", np.round(repeated_scores_a3, 3)) # inspect all runs.

▶ What you'll see: four slightly different outer-score rows.

In [ ]:
run_means_a3 = np.mean(repeated_scores_a3, axis=1) # nested estimate per repeat.
overall_a3 = float(np.mean(run_means_a3)) # repeated nested CV average.
spread_a3 = float(np.std(run_means_a3, ddof=1)) # variability across repeats.
print("run means:", np.round(run_means_a3, 3)) # inspect repeated estimates.
print("overall mean:", round(overall_a3, 3), "spread:", round(spread_a3, 3)) # inspect summary.
assert round(overall_a3, 3) == 0.326 # verify stable average.

In [ ]:
plt.figure(figsize=(5, 3)) # create a repeated-CV plot.
plt.plot(np.arange(1, 5), run_means_a3, marker="o", color="purple") # show each repeated estimate.
plt.axhline(overall_a3, color="black", linestyle="--", label=f"overall={overall_a3:.3f}") # show repeated average.
plt.title("Advanced 3: repeated nested estimates") # title the plot.
plt.xlabel("repeat") # label repeat axis.
plt.ylabel("nested estimate") # label score axis.
plt.legend() # show average label.
plt.show() # display the plot.

▶ What you'll see: repeated estimates wiggle slightly around the overall mean.

In [ ]:
best_gap_a3 = 0.044 # lesson's baseline-vs-flexible gap.
print("gap / repeat spread:", round(best_gap_a3 / spread_a3, 1)) # compare effect size to split noise.
assert best_gap_a3 / spread_a3 > 10 # this toy gap is large relative to repeat spread.

▶ What you'll see: the toy gap is much larger than this repeated-split spread.

👀 Takeaway: repeated nested CV helps distinguish real improvements from split-specific noise.

### Advanced 4 — One-standard-error style stabilization

**Goal.** Prefer a simpler/stabler setting when its score is statistically close to the minimum, because tiny validation wins often do not justify extra flexibility. We build it in 4 steps.

In [ ]:
lambdas_a4 = np.array([0.0, 0.1, 1.0, 10.0]) # larger λ means more stabilization.
mean_scores_a4 = np.array([0.318, 0.312, 0.320, 0.360]) # inner CV mean losses.
se_scores_a4 = np.array([0.012, 0.010, 0.009, 0.014]) # standard errors for the means.
print("mean scores:", mean_scores_a4) # inspect candidate means.
print("standard errors:", se_scores_a4) # inspect uncertainty.

▶ What you'll see: λ=0.1 has the lowest mean, but λ=1.0 is close.

In [ ]:
best_idx_a4 = int(np.argmin(mean_scores_a4)) # minimum mean validation score.
threshold_a4 = mean_scores_a4[best_idx_a4] + se_scores_a4[best_idx_a4] # one-SE threshold.
eligible_a4 = mean_scores_a4 <= threshold_a4 # candidates close enough to the minimum.
stable_idx_a4 = int(np.max(np.where(eligible_a4)[0])) # choose the largest λ among eligible settings.
print("best λ by mean:", lambdas_a4[best_idx_a4]) # inspect raw winner.
print("one-SE threshold:", round(float(threshold_a4), 3)) # inspect tolerance.
print("one-SE stabilized λ:", lambdas_a4[stable_idx_a4]) # inspect stable choice.
assert lambdas_a4[stable_idx_a4] == 1.0 # verify the simpler close-enough choice.

In [ ]:
plt.figure(figsize=(5, 3)) # create a one-SE plot.
plt.errorbar(lambdas_a4, mean_scores_a4, yerr=se_scores_a4, fmt="o-", color="navy", capsize=4) # plot means with uncertainty.
plt.axhline(threshold_a4, color="red", linestyle="--", label="one-SE threshold") # show allowed band.
plt.axvline(lambdas_a4[stable_idx_a4], color="green", linestyle=":", label="stable choice") # mark stabilized λ.
plt.xscale("symlog") # show zero and large λ on a readable axis.
plt.title("Advanced 4: one-SE stabilization") # title the plot.
plt.xlabel("λ") # label regularization axis.
plt.ylabel("inner CV loss") # label loss axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: λ=1.0 lies within the acceptable band and is more stable than λ=0.1.

In [ ]:
score_drop_a4 = 0.80 * 0.320 # connect the stability idea to the lesson's 20% score reduction.
print("lesson stabilized score:", round(score_drop_a4, 3)) # inspect 0.256.
assert round(score_drop_a4, 3) == 0.256 # verify the lesson's stabilized value.

▶ What you'll see: the lesson's stabilized score is `0.256` after the 20% reduction.

👀 Takeaway: stabilization rules trade tiny mean-score wins for choices that are less likely to be brittle.

### Advanced 5 — Compare model families with an outer decision table

**Goal.** Evaluate full model-selection procedures side by side, because nested CV estimates workflows, not isolated fitted models. We build it in 4 steps.

In [ ]:
families_a5 = np.array(["linear", "polynomial", "stabilized"] ) # candidate workflows.
outer_table_a5 = np.array([[0.342, 0.331, 0.310], [0.356, 0.344, 0.322], [0.330, 0.337, 0.300], [0.348, 0.352, 0.318]]) # outer losses by fold and workflow.
print("outer decision table:\n", outer_table_a5) # inspect held-out losses.
print("families:", families_a5) # inspect workflow names.

▶ What you'll see: each workflow has one held-out loss per outer fold.

In [ ]:
mean_family_a5 = np.mean(outer_table_a5, axis=0) # average outer loss per workflow.
se_family_a5 = np.std(outer_table_a5, axis=0, ddof=1) / np.sqrt(outer_table_a5.shape[0]) # uncertainty per workflow mean.
best_family_idx_a5 = int(np.argmin(mean_family_a5)) # choose the lowest outer mean.
print("mean outer loss:", dict(zip(families_a5, np.round(mean_family_a5, 3)))) # inspect means.
print("best workflow:", families_a5[best_family_idx_a5]) # inspect winner.
assert families_a5[best_family_idx_a5] == "stabilized" # verify stabilized workflow wins here.

In [ ]:
margin_a5 = float(mean_family_a5[1] - mean_family_a5[2]) # margin over the polynomial workflow.
print("margin vs polynomial:", round(margin_a5, 3)) # inspect the decision gap.
assert round(margin_a5, 3) == 0.028 # verify concrete margin.

In [ ]:
plt.figure(figsize=(5, 3)) # create a workflow comparison chart.
plt.bar(families_a5, mean_family_a5, yerr=se_family_a5, capsize=5, color=["gray", "orange", "seagreen"]) # compare mean outer losses with uncertainty.
plt.title("Advanced 5: compare full workflows") # title the plot.
plt.ylabel("mean outer loss") # label loss axis.
plt.xticks(rotation=10) # keep labels readable.
plt.show() # display the plot.

▶ What you'll see: the stabilized workflow has the lowest mean outer loss, with uncertainty bars for context.

👀 Takeaway: the correct comparison unit is the whole training-and-selection workflow evaluated on outer folds.